# Разбор A. Энсамбль, низкие частоты и пределы численного ранга

Один минимум не определяет распределение конформеров. Для трёх Hessian-проверенных структур истрадефиллина сравним restricted weights по electronic+ALPB energies и native model free-energy values. Полная функция распределения по неизвестным состояниям и экспериментальные populations отсюда не следуют.

Термодинамическая формула w_i∝g_i exp(−ΔG_i/RT) требует свободных энергий, degeneracy и полноты состояний. Здесь degeneracy не установлена, набор ограничен тремя конформерами, а native thermochemistry использует собственную coupled RRHO/low-mode модель xTB при 298.15 K. Её значения сохраняются как model-dependent diagnostic, не как проверенная solution free energy.

In [2]:
from pathlib import Path
import sys,json,hashlib,math
import numpy as np
from scipy import stats
from rdkit import Chem
from rdkit.Chem import rdMolAlign
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUT=workspace('M06-L02','case');C_DATA=DATA/'conformer_research'
manifest=json.loads((C_DATA/'bundle-manifest.json').read_text(encoding='utf-8'))
for name,digest in manifest['files_sha256'].items():assert hashlib.sha256((C_DATA/name).read_bytes()).hexdigest()==digest,name
def read(state,name):return json.loads((C_DATA/state/name).read_text(encoding='utf-8'))
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=150,bbox_inches='tight');display(fig);plt.close(fig)


In [3]:
state='istradefylline';report=read(state,'result.json');checked=report['hessian_results']
assert all(r['status']=='PASS' and r['minimum_verified'] for r in checked)
ids=[r['source_id'] for r in checked]
native=[read(state,f'hessian_{i:04}_native.json') for i in ids]
E=np.array([r['energy_hartree'] for r in checked]);G=np.array([r['total free energy'] for r in native])
T=298.15;R_kcal=8.31446261815324/4184
def restricted_weights(energy_Eh):
    relative=(energy_Eh-energy_Eh.min())*627.5094740631
    weight=np.exp(-relative/(R_kcal*T))
    return weight/weight.sum()
wE=restricted_weights(E);wG=restricted_weights(G)
display([{'source_id':i,'model_E_relative_kcal_mol':float(e),'model_G_relative_kcal_mol':float(g),
          'restricted_E_weight':float(a),'restricted_native_G_weight':float(b),'lowest_internal_frequency_cm1':r['lowest_frequency_cm1']}
         for i,e,g,a,b,r in zip(ids,(E-E.min())*627.5094740631,(G-G.min())*627.5094740631,wE,wG,checked)])
fig,ax=plt.subplots();x=np.arange(len(ids));ax.bar(x-.18,wE,.36,label='E + ALPB, restricted set');ax.bar(x+.18,wG,.36,label='Native model G, restricted set')
ax.set_xticks(x,[str(i) for i in ids]);ax.set(xlabel='Source conformer ID',ylabel='Restricted model weight');ax.legend(fontsize=8)
publish_plot(fig,'restricted_ensemble_weights.png')


[{'source_id': 172,
  'model_E_relative_kcal_mol': 0.0,
  'model_G_relative_kcal_mol': 0.001964104653828698,
  'restricted_E_weight': 0.3345768346029273,
  'restricted_native_G_weight': 0.3377320050449257,
  'lowest_internal_frequency_cm1': 12.068262273303336},
 {'source_id': 151,
  'model_E_relative_kcal_mol': 0.0005134320105052798,
  'model_G_relative_kcal_mol': 0.0,
  'restricted_E_weight': 0.33428702462996895,
  'restricted_native_G_weight': 0.33885345414906487,
  'lowest_internal_frequency_cm1': 11.7319919363968},
 {'source_id': 293,
  'model_E_relative_kcal_mol': 0.00612449371374665,
  'model_G_relative_kcal_mol': 0.027629242143015997,
  'restricted_E_weight': 0.33113614076710374,
  'restricted_native_G_weight': 0.32341454080600945,
  'lowest_internal_frequency_cm1': 11.87740612061137}]

<Figure size 640x480 with 1 Axes>

Проверим численную устойчивость до физической интерпретации точного порядка. Парные GFN2 calculations при той же геометрии отделяют модельную solvation contribution от gas energy. Более плотная SASA-сетка и вдвое меньший Hessian step проверяют разные численные источники.

In [4]:
audit=read(state,'numerics.json');pairs=audit['paired_fixed_geometry_results']
display(pairs)
order_tight=np.argsort([r['tight_energy_Eh'] for r in pairs]).tolist()
order_dense=np.argsort([r['verytight_energy_Eh'] for r in pairs]).tolist()
print({'grid_order_tight':order_tight,'grid_order_verytight':order_dense,
       'relative_energy_grid_sensitivity_kcal_mol':audit['max_relative_energy_grid_sensitivity_kcal_mol'],
       'Hessian_half_step_max_frequency_change_cm1':audit['Hessian_step_max_frequency_change_cm1'],
       'minimum_sign_stable':audit['minimum_sign_stable']})
result={'kind':'calculated','scope':'restricted saved-ensemble and numerical-sensitivity analysis','conformer_ids':ids,
        'restricted_E_weights':wE.tolist(),'restricted_native_G_weights':wG.tolist(),
        'precise_ordering_accepted':order_tight==order_dense,'complete_population_claim_allowed':False,
        'DFT_validation':'separate and incomplete','new_external_calculation':'NOT_RUN'}


[{'source_id': 172,
  'tight_energy_Eh': -83.149604357449,
  'verytight_energy_Eh': -83.149603982184,
  'gas_energy_Eh': -83.126906985342,
  'fixed_geometry_ALPB_minus_gas_Eh': -0.022697372106989633,
  'surface_grid_energy_change_Eh': 3.752649888610904e-07},
 {'source_id': 151,
  'tight_energy_Eh': -83.149603539243,
  'verytight_energy_Eh': -83.149604667864,
  'gas_energy_Eh': -83.126904512221,
  'fixed_geometry_ALPB_minus_gas_Eh': -0.022699027021999996,
  'surface_grid_energy_change_Eh': -1.128620993995355e-06},
 {'source_id': 293,
  'tight_energy_Eh': -83.149594597447,
  'verytight_energy_Eh': -83.149594073191,
  'gas_energy_Eh': -83.126895058707,
  'fixed_geometry_ALPB_minus_gas_Eh': -0.022699538739999525,
  'surface_grid_energy_change_Eh': 5.242560092710846e-07}]

{'grid_order_tight': [0, 1, 2], 'grid_order_verytight': [1, 0, 2], 'relative_energy_grid_sensitivity_kcal_mol': 0.0010371959790107163, 'Hessian_half_step_max_frequency_change_cm1': 0.1834445724693552, 'minimum_sign_stable': True}


Для близкой пары уточнение сетки меняет порядок: точные последние знаки не доказывают предпочтение одного конформера. При этом Hessian frequencies изменились максимум на ≈0.183 cm⁻¹ и положительный знак сохранился. Эти проверки оценивают numerical sensitivity, не все ошибки Hamiltonian/solvent/entropy model. Следующий шаг — дополнительные состояния, low-mode/degeneracy sensitivity, целевые DFT и независимые solution-state данные, например подходящие NMR ограничения.

In [5]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
print(append_record(init_dossier(),'M06-L02','A',result,ROOT,inputs=['practicuum/shared/data/conformer_research/atorvastatin/geometry_0013.xyz', 'practicuum/shared/data/conformer_research/atorvastatin/geometry_0112.xyz', 'practicuum/shared/data/conformer_research/atorvastatin/geometry_0198.xyz', 'practicuum/shared/data/conformer_research/atorvastatin/hessian_0013_native.json', 'practicuum/shared/data/conformer_research/atorvastatin/hessian_0013_projected.json', 'practicuum/shared/data/conformer_research/atorvastatin/hessian_0112_native.json', 'practicuum/shared/data/conformer_research/atorvastatin/hessian_0112_projected.json', 'practicuum/shared/data/conformer_research/atorvastatin/hessian_0198_native.json', 'practicuum/shared/data/conformer_research/atorvastatin/hessian_0198_projected.json', 'practicuum/shared/data/conformer_research/atorvastatin/manifest.json', 'practicuum/shared/data/conformer_research/atorvastatin/numerics.json', 'practicuum/shared/data/conformer_research/atorvastatin/optimization_selection.json', 'practicuum/shared/data/conformer_research/atorvastatin/optimized_conformers.sdf', 'practicuum/shared/data/conformer_research/atorvastatin/result.json', 'practicuum/shared/data/conformer_research/atorvastatin/singlepoint_results.json', 'practicuum/shared/data/conformer_research/atorvastatin_anion/geometry_0126.xyz', 'practicuum/shared/data/conformer_research/atorvastatin_anion/geometry_0163.xyz', 'practicuum/shared/data/conformer_research/atorvastatin_anion/geometry_0206.xyz', 'practicuum/shared/data/conformer_research/atorvastatin_anion/hessian_0126_native.json', 'practicuum/shared/data/conformer_research/atorvastatin_anion/hessian_0126_projected.json', 'practicuum/shared/data/conformer_research/atorvastatin_anion/hessian_0163_native.json', 'practicuum/shared/data/conformer_research/atorvastatin_anion/hessian_0163_projected.json', 'practicuum/shared/data/conformer_research/atorvastatin_anion/hessian_0206_native.json', 'practicuum/shared/data/conformer_research/atorvastatin_anion/hessian_0206_projected.json', 'practicuum/shared/data/conformer_research/atorvastatin_anion/manifest.json', 'practicuum/shared/data/conformer_research/atorvastatin_anion/optimization_selection.json', 'practicuum/shared/data/conformer_research/atorvastatin_anion/optimized_conformers.sdf', 'practicuum/shared/data/conformer_research/atorvastatin_anion/result.json', 'practicuum/shared/data/conformer_research/atorvastatin_anion/singlepoint_results.json', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/geometry_0081.xyz', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/geometry_0199.xyz', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/geometry_0265.xyz', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/hessian_0081_native.json', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/hessian_0081_projected.json', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/hessian_0199_native.json', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/hessian_0199_projected.json', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/hessian_0265_native.json', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/hessian_0265_projected.json', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/manifest.json', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/optimization_selection.json', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/optimized_conformers.sdf', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/result.json', 'practicuum/shared/data/conformer_research/atorvastatin_lactone/singlepoint_results.json', 'practicuum/shared/data/conformer_research/bundle-manifest.json', 'practicuum/shared/data/conformer_research/istradefylline/geometry_0151.xyz', 'practicuum/shared/data/conformer_research/istradefylline/geometry_0172.xyz', 'practicuum/shared/data/conformer_research/istradefylline/geometry_0293.xyz', 'practicuum/shared/data/conformer_research/istradefylline/hessian_0151_native.json', 'practicuum/shared/data/conformer_research/istradefylline/hessian_0151_projected.json', 'practicuum/shared/data/conformer_research/istradefylline/hessian_0172_native.json', 'practicuum/shared/data/conformer_research/istradefylline/hessian_0172_projected.json', 'practicuum/shared/data/conformer_research/istradefylline/hessian_0293_native.json', 'practicuum/shared/data/conformer_research/istradefylline/hessian_0293_projected.json', 'practicuum/shared/data/conformer_research/istradefylline/manifest.json', 'practicuum/shared/data/conformer_research/istradefylline/numerics.json', 'practicuum/shared/data/conformer_research/istradefylline/optimization_selection.json', 'practicuum/shared/data/conformer_research/istradefylline/optimized_conformers.sdf', 'practicuum/shared/data/conformer_research/istradefylline/result.json', 'practicuum/shared/data/conformer_research/istradefylline/singlepoint_results.json']))


C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261119-professional-conformers-windows\workspace\my_research\evidence\M06-L02\20261119T084633785853Z-A-ae57b32c8778424a8c4d19cfa2b48803.json
